# Turing Remote Runbook

Use this notebook from VS Code Remote on Turing as a lightweight viewer and launchpad. Heavy compute should run via `sbatch`/`torchrun`, not inside notebook cells.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path(os.environ.get("FRM_REPO_DIR", "")).expanduser() if os.environ.get("FRM_REPO_DIR") else None
if ROOT is None or not (ROOT / "configs" / "default.toml").exists():
    candidates = [
        Path.cwd(),
        Path(f"/local/scratch/{os.environ.get('USER', '')}/forward-risk-manager/repo"),
        Path(f"/local/scratch/{os.environ.get('USER', '')}/forward-risk-manager"),
    ]
    ROOT = next((p.resolve() for p in candidates if (p / "configs" / "default.toml").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Set FRM_REPO_DIR to the repo on scratch before using this notebook.")
sys.path.append(str(ROOT / "src"))
from frisk.notebook_runtime import load_json, resolve_repo_root

ROOT = resolve_repo_root(start=ROOT)
SCRATCH_ROOT = ROOT.parent if ROOT.name == "repo" else ROOT
RUNS_ROOT = SCRATCH_ROOT / "runs"
LOGS_ROOT = SCRATCH_ROOT / "logs"
ROOT, SCRATCH_ROOT, RUNS_ROOT, LOGS_ROOT

In [ ]:
commands = {
    "prep": "python scripts/prepare_turing_run.py --base-config configs/default.toml --cluster-config configs/turing.toml --runtime-config /local/scratch/$USER/forward-risk-manager/runs/runtime_turing.toml --netid $USER",
    "train": "sbatch slurm/train_2gpu.sbatch",
    "benchmark": "sbatch slurm/benchmark_2gpu.sbatch",
    "sweep": "sbatch slurm/sweep_2gpu.sbatch",
    "shard": "sbatch slurm/shard_graph.sbatch",
    "smoke": "sbatch slurm/smoke_test.sbatch",
}
commands

In [ ]:
manifest_candidates = sorted(RUNS_ROOT.rglob("manifest.json"))
latest_manifest = manifest_candidates[-1] if manifest_candidates else None
latest_manifest, load_json(latest_manifest, default={}) if latest_manifest else {}